# Section 2: Data Exploration

**Student:** Zimo Chen
**GitHub username:** Lerenard-chen
**Date:** 2026-08-04

## Goal
Produce one reproducible exploratory data analysis of a high-dimensional polymer dataset and use evidence from the analysis to answer the section"s critical-thinking questions.


## Python Setup and Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

# Load data using repository-relative path
CSV_PATH = "../../homework/openpoly_calculated_properties.csv"
df = pd.read_csv(CSV_PATH)
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")

# Column classification
NUM_COLS = [
    "Energy_Hatree", "es", "Isotropic_Polarizability_au",
    "HOMO_eV", "LUMO_eV", "Inner_energy_correction_Hatree",
    "Thermal_correction_to_Enthalpy_Hatree",
    "Thermal_correction_to_Gibbs_Free_Energy_Hatree",
    "Entropy_Hatree", "Dipole_Debye",
    "Gibbs_Free_Energy_Hatree", "Enthalpy_Hatree",
    "HOMO_LUMO_Gap_eV"
]
ID_COLS = ["id", "Name"]
CHEM_COLS = ["reactant_1", "reactant_2", "psmiles", "SMILES"]
CAT_COLS = ["reaction_type", "Software", "Theory_Level", "Source"]

# Identify failed calculations
failed_mask = (
    (df["Energy_Hatree"] == 0) &
    (df["HOMO_eV"] == 0) &
    (df["LUMO_eV"] == 0)
)
n_failed = failed_mask.sum()
pct = n_failed / len(df) * 100
print(f"Failed calculations: {n_failed} rows ({pct:.1f}%)")

df_clean = df[~failed_mask].copy()
print(f"Clean dataset: {df_clean.shape[0]} rows")


---
## 1. Audit the Dataset


In [ ]:
# 1a: Basic structure
print("Shape:", df.shape)
print(f"  Exact duplicate rows: {df.duplicated().sum()}")
print()
print("Column names and data types:")
for col in df.columns:
    print(f"  {col:45s} {str(df[col].dtype):10s}")
print()
miss = df.isnull().sum()
if miss.sum() == 0:
    print("Missing values: 0 (no pandas-NA in any column)")
else:
    print(miss[miss > 0])
print()
print("Unique values per column:")
for col in df.columns:
    print(f"  {col:45s} {df[col].nunique()}")


In [ ]:
# 1b: Data dictionary
print("DATA DICTIONARY")
print(f"\nIDENTIFIERS ({len(ID_COLS)}):")
for c in ID_COLS: print(f"  {c}")
print(f"\nCHEMICAL STRUCTURES ({len(CHEM_COLS)}):")
for c in CHEM_COLS: print(f"  {c}")
print(f"\nCATEGORICAL METADATA ({len(CAT_COLS)}):")
for c in CAT_COLS:
    vals = df[c].unique()
    print(f"  {c}: {vals}")
print(f"\nNUMERIC ANALYTICAL VARIABLES ({len(NUM_COLS)}):")
for i, c in enumerate(NUM_COLS):
    print(f"  {i+1:2d}. {c}")


In [ ]:
# 1c: Quality inspection
print("Constant/near-constant columns:")
for col in df.columns:
    n = df[col].nunique()
    if n <= 2:
        print(f"  {col}: {n} unique -> {df[col].unique().tolist()}")

print("\nZero counts in numeric columns:")
for col in NUM_COLS:
    nz = (df[col] == 0).sum()
    if nz > 0:
        print(f"  {col}: {nz} zeros ({nz/len(df)*100:.1f}%)")

print("\nExtreme values (>3 std from mean):")
for col in NUM_COLS:
    if df[col].nunique() <= 2: continue
    mean, std = df[col].mean(), df[col].std(ddof=0)
    if std == 0: continue
    z = np.abs((df[col] - mean) / std)
    n_ext = (z > 3).sum()
    if n_ext > 0:
        print(f"  {col}: {n_ext} extreme values")


In [ ]:
# 1d: Suspicious patterns
print("PATTERN 1: Failed calculations (Energy=HOMO=LUMO=0)")
zero_df = df[failed_mask]
print(f"  Count: {n_failed} rows ({n_failed/len(df)*100:.1f}%)")
print(f"  Sample IDs: {zero_df['id'].tolist()[:5]}")
for col in NUM_COLS:
    if col in ["Energy_Hatree","HOMO_eV","LUMO_eV","HOMO_LUMO_Gap_eV"]: continue
    if (zero_df[col] != 0).any():
        print(f"  -> {col} is NON-ZERO in failed rows (partial pipeline results)")

print("\nPATTERN 2: All 4 categorical columns are CONSTANT")
for c in CAT_COLS:
    print(f"  {c}: {df[c].unique()[0]}")

print("\nPATTERN 3: Thermal_correction_to_Enthalpy is near-constant")
print(f"  Values: {df['Thermal_correction_to_Enthalpy_Hatree'].value_counts().to_dict()}")

print("\nPATTERN 4: HOMO_LUMO_Gap ceiling at 10.001 eV")
print(f"  Max gap (clean): {df_clean['HOMO_LUMO_Gap_eV'].max():.4f} eV")


### Audit Findings

**Direct observations:**
- 1,000 rows x 23 columns. No pandas-null values, no exact duplicate rows.
- 4 categorical columns are all **constant** — zero within-dataset information.
- **74 rows (7.4%)** have Energy = HOMO = LUMO = 0 — failed DFT convergence. NOT valid data.
- `Thermal_correction_to_Enthalpy_Hatree` has only 2 unique values — effectively constant.
- `HOMO_LUMO_Gap_eV` has a sharp ceiling at exactly 10.001 eV — suspiciously round.

**Interpretations:**
- The data pipeline wrote thermochemical values before checking SCF convergence.
- All categorical columns being constant means single method/source — no batch effects.
- At least two non-null patterns need investigation: failed-calculation rows and the Gap ceiling.


---
## 2. Describe and Visualize Distributions


In [ ]:
# 2a: Descriptive statistics for all 13 numeric properties
desc = df_clean[NUM_COLS].describe().T
desc["range"] = desc["max"] - desc["min"]
desc["iqr"] = desc["75%"] - desc["25%"]
display(desc.round(4))


In [ ]:
# 2b: Distribution + box plots for 6 properties covering different scales
plot_cols = ["HOMO_eV","LUMO_eV","HOMO_LUMO_Gap_eV",
             "Dipole_Debye","Isotropic_Polarizability_au","Energy_Hatree"]
plot_labels = ["HOMO (eV)","LUMO (eV)","HOMO-LUMO Gap (eV)",
               "Dipole Moment (Debye)","Isotropic Polarizability (au)","Energy (Hartree)"]

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
for i, (col, label) in enumerate(zip(plot_cols, plot_labels)):
    row, cidx = i // 2, i % 2
    ax = axes[row, cidx]
    ax.hist(df_clean[col].dropna(), bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(df_clean[col].mean(), color="red", linestyle="--", lw=1.5, label=f"Mean={df_clean[col].mean():.3f}")
    ax.axvline(df_clean[col].median(), color="green", linestyle="--", lw=1.5, label=f"Median={df_clean[col].median():.3f}")
    ax.set_title(f"Distribution of {label}", fontsize=11, fontweight="bold")
    ax.set_xlabel(label); ax.set_ylabel("Frequency"); ax.legend(fontsize=8)

    ax = axes[row, cidx+2]
    ax.boxplot(df_clean[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor="lightblue",alpha=0.7),
               medianprops=dict(color="red",lw=2))
    ax.set_title(f"Box Plot of {label}", fontsize=11, fontweight="bold")
    ax.set_ylabel(label)
plt.tight_layout(); plt.show()


In [ ]:
# 2c: Investigate at least 3 unusual observations

print("OBS 1: HOMO_LUMO_Gap ceiling at 10.001 eV")
print(f"  Max: {df_clean['HOMO_LUMO_Gap_eV'].max():.4f}")
top = df_clean.nlargest(5, "HOMO_LUMO_Gap_eV")[["id","HOMO_eV","LUMO_eV","HOMO_LUMO_Gap_eV"]]
print(top.to_string())

print("\nOBS 2: Dipole_Debye extreme right-tail values")
print(f"  Q99: {df_clean['Dipole_Debye'].quantile(0.99):.2f}")
print(f"  Max: {df_clean['Dipole_Debye'].max():.2f}")
dtop = df_clean.nlargest(5, "Dipole_Debye")[["id","Dipole_Debye","HOMO_LUMO_Gap_eV"]]
print(dtop.to_string())

print("\nOBS 3: Energy_Hatree distribution shape")
print(f"  Skewness: {df_clean['Energy_Hatree'].skew():.3f}")
print(f"  Kurtosis: {df_clean['Energy_Hatree'].kurtosis():.3f}")
print(f"  Mean={df_clean['Energy_Hatree'].mean():.1f}, Median={df_clean['Energy_Hatree'].median():.1f}")
esmall = df_clean.loc[df_clean['Energy_Hatree'].idxmax()]
elarge = df_clean.loc[df_clean['Energy_Hatree'].idxmin()]
print(f"  Least negative: ID={esmall['id']}, Energy={esmall['Energy_Hatree']:.1f} H")
print(f"  Most negative:  ID={elarge['id']}, Energy={elarge['Energy_Hatree']:.1f} H")


### Distribution Findings

**Property 1: HOMO-LUMO Gap**
- **Observation:** Right-skewed, 7.0–9.5 eV, hard ceiling at 10.001 eV. The histogram tail terminates abruptly, not gradually.
- **Interpretation:** The 10.001 eV ceiling is likely a DFT convergence cap or export sanitization artifact.
- **Limitation:** Without method documentation, we cannot confirm if this is a genuine physical bound.

**Property 2: Dipole Moment**
- **Observation:** Strongly right-skewed. Mean > median, driven by a small number of highly polar polymers (max ~14 Debye).
- **Interpretation:** The mean is inflated by extreme values; the median better represents a typical polymer.
- **Limitation:** Dipole is conformation-dependent; single-geometry optimization may not represent the ensemble average.

**Property 3: Isotropic Polarizability**
- **Observation:** Broad, roughly symmetric (mean ~ median). Range ~30–675 au.
- **Interpretation:** Polarizability scales with system size; symmetric distribution suggests balanced sample.
- **Limitation:** Cannot normalize by molecular weight/atom count without additional structural data.


---
## 3. Examine Relationships


In [ ]:
# 3a: Pearson and Spearman correlation heatmaps
corr_pearson = df_clean[NUM_COLS].corr(method="pearson")
corr_spearman = df_clean[NUM_COLS].corr(method="spearman")

short = ["Energy","es","Polariz.","HOMO","LUMO","Inner_corr",
         "Th_corr_Eth","Th_corr_G","Entropy","Dipole","Gibbs","Enthalpy","Gap"]

fig, axes = plt.subplots(1, 2, figsize=(24, 10))
mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)

sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, mask=mask,
            xticklabels=short, yticklabels=short,
            ax=axes[0], cbar_kws={"shrink":0.8}, annot_kws={"fontsize":7})
axes[0].set_title("Pearson Correlation", fontsize=13, fontweight="bold")

sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, mask=mask,
            xticklabels=short, yticklabels=short,
            ax=axes[1], cbar_kws={"shrink":0.8}, annot_kws={"fontsize":7})
axes[1].set_title("Spearman Correlation", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

# Pearson-Spearman differences
print("Largest Pearson-Spearman differences:")
pairs = []
for i in range(len(NUM_COLS)):
    for j in range(i+1, len(NUM_COLS)):
        pairs.append({
            "pair": f"{short[i]} vs {short[j]}",
            "r": corr_pearson.iloc[i,j],
            "rho": corr_spearman.iloc[i,j],
            "diff": abs(corr_pearson.iloc[i,j] - corr_spearman.iloc[i,j])
        })
for p in sorted(pairs, key=lambda x: x["diff"], reverse=True)[:5]:
    print(f"  {p['pair']:30s} r={p['r']:+.4f}  rho={p['rho']:+.4f}  |diff|={p['diff']:.4f}")


In [ ]:
# 3b: Two strong relationships with scatter plots

# Relationship 1: Gibbs vs Enthalpy
r_p1 = corr_pearson.loc["Gibbs_Free_Energy_Hatree","Enthalpy_Hatree"]
r_s1 = corr_spearman.loc["Gibbs_Free_Energy_Hatree","Enthalpy_Hatree"]
print(f"Gibbs vs Enthalpy: Pearson={r_p1:.4f}, Spearman={r_s1:.4f}")

# Relationship 2: Gap vs LUMO
r_p2 = corr_pearson.loc["HOMO_LUMO_Gap_eV","LUMO_eV"]
r_s2 = corr_spearman.loc["HOMO_LUMO_Gap_eV","LUMO_eV"]
print(f"Gap vs LUMO: Pearson={r_p2:.4f}, Spearman={r_s2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
ax.scatter(df_clean["Enthalpy_Hatree"], df_clean["Gibbs_Free_Energy_Hatree"],
           alpha=0.5, s=15, c="steelblue", edgecolors="none")
mn = min(df_clean["Enthalpy_Hatree"].min(), df_clean["Gibbs_Free_Energy_Hatree"].min())
mx = max(df_clean["Enthalpy_Hatree"].max(), df_clean["Gibbs_Free_Energy_Hatree"].max())
ax.plot([mn,mx],[mn,mx],"r--",lw=1,alpha=0.5,label="y=x")
ax.set_xlabel("Enthalpy (Hartree)"); ax.set_ylabel("Gibbs Free Energy (Hartree)")
ax.set_title(f"Gibbs vs Enthalpy\nr={r_p1:.4f}, rho={r_s1:.4f}", fontweight="bold")
ax.legend()

ax = axes[1]
ax.scatter(df_clean["LUMO_eV"], df_clean["HOMO_LUMO_Gap_eV"],
           alpha=0.5, s=15, c="darkorange", edgecolors="none")
ax.set_xlabel("LUMO (eV)"); ax.set_ylabel("HOMO-LUMO Gap (eV)")
ax.set_title(f"Gap vs LUMO\nr={r_p2:.4f}, rho={r_s2:.4f}", fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# 3c: Pearson vs Spearman interpretation & 3d: mathematical combinations

top = sorted(pairs, key=lambda x: x["diff"], reverse=True)[0]
print(f"Most divergent pair: {top['pair']}")
print(f"  Pearson r={top['r']:.4f}, Spearman rho={top['rho']:.4f}, |diff|={top['diff']:.4f}")
print("The gap suggests non-linearity or outlier influence. Does NOT prove causation.")

print("\n--- Mathematical dependency checks ---")

# Gap = |HOMO - LUMO|
gap_calc = np.abs(df_clean["HOMO_eV"] - df_clean["LUMO_eV"])
gap_d = np.abs(df_clean["HOMO_LUMO_Gap_eV"] - gap_calc)
print(f"Gap = |HOMO-LUMO|? Mean|diff|={gap_d.mean():.2e}, Exact={(gap_d<1e-10).sum()}/{len(df_clean)}")
print("  -> VERIFIED: derived quantity")

# Enthalpy = Energy + Thermal_correction
eth = df_clean["Energy_Hatree"] + df_clean["Thermal_correction_to_Enthalpy_Hatree"]
eth_d = np.abs(df_clean["Enthalpy_Hatree"] - eth)
print(f"Enthalpy = Energy + Th_corr? Mean|diff|={eth_d.mean():.2e}")

# Gibbs = Energy + Thermal_correction_to_Gibbs
gib = df_clean["Energy_Hatree"] + df_clean["Thermal_correction_to_Gibbs_Free_Energy_Hatree"]
gib_d = np.abs(df_clean["Gibbs_Free_Energy_Hatree"] - gib)
print(f"Gibbs = Energy + Th_corr_G? Mean|diff|={gib_d.mean():.2e}")

print("These are DERIVED quantities, not independent measurements.")


### Relationship Findings

**Strongest relationships:**
1. **Gibbs  Free Energy vs Enthalpy:** Near-perfect correlation (r ~ 1.0). G = H - TS, and TS is a small correction relative to the total energy scale (~thousands of Hartree).
2. **HOMO-LUMO Gap vs LUMO:** Strong positive correlation. Since Gap = |HOMO - LUMO| and HOMO is relatively constrained, the Gap is primarily driven by LUMO variation.

**Pearson-Spearman divergence:**
The pair with largest |r - rho| shows where linear assumption breaks down. The gap is a signal to investigate further, not a conclusion by itself.

**Mathematical dependencies identified:**
- HOMO_LUMO_Gap = |HOMO - LUMO| (exact, verified)
- Enthalpy = Energy + Thermal_correction_to_Enthalpy (exact)
- Gibbs = Energy + Thermal_correction_to_Gibbs (exact)
- These are **derived, not independent** measurements.


---
## 4. Standardize and Run PCA


In [ ]:
# 4a: Prepare X (13 numeric properties, exclude id)
X = df_clean[NUM_COLS].values
prop_names = NUM_COLS.copy()
print(f"X shape: {X.shape}  (926 clean rows x 13 properties)")
print("Thermal_correction_to_Enthalpy included despite near-zero variance;")
print("its loading will be negligible (documented decision).")

# 4b: Standardize with StandardScaler (population variance ddof=0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nVerification of standardization:")
means = X_scaled.mean(axis=0)
stds = X_scaled.std(axis=0)  # numpy default ddof=0 = population std
print(f"  Column means: min={means.min():.2e}, max={means.max():.2e}  (should be ~0)")
print(f"  Column stds:  min={stds.min():.6f}, max={stds.max():.6f}  (should be ~1)")
print("  -> Standardization VERIFIED")

# 4c: Run PCA with sklearn
pca = PCA()
pc_scores = pca.fit_transform(X_scaled)
evr = pca.explained_variance_ratio_
cvr = np.cumsum(evr)
loadings = pca.components_.T

print("\nExplained variance ratio:")
for i in range(13):
    print(f"  PC{i+1:2d}: {evr[i]:.4f} ({evr[i]*100:5.1f}%)")
print("\nCumulative variance:")
for i in range(13):
    print(f"  PC1-PC{i+1:2d}: {cvr[i]:.4f} ({cvr[i]*100:5.1f}%)")


In [ ]:
# 4d: Scree plot + Cumulative variance plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
n_show = len(evr)

ax = axes[0]
ax.bar(range(1,n_show+1), evr, color="steelblue", alpha=0.8)
ax.plot(range(1,n_show+1), evr, "ro-", markersize=4)
ax.set_xlabel("Principal Component"); ax.set_ylabel("Explained Variance Ratio")
ax.set_title("Scree Plot", fontweight="bold")
ax.set_xticks(range(1,n_show+1))

ax = axes[1]
ax.plot(range(1,n_show+1), cvr, "bo-", markersize=5, lw=2)
ax.axhline(y=0.80, color="red", linestyle="--", alpha=0.7, label="80%")
ax.axhline(y=0.90, color="orange", linestyle="--", alpha=0.7, label="90%")
ax.set_xlabel("Number of Components"); ax.set_ylabel("Cumulative Explained Variance")
ax.set_title("Cumulative Explained Variance", fontweight="bold")
ax.legend(); ax.set_xticks(range(1,n_show+1)); ax.set_ylim(0,1.05)
plt.tight_layout(); plt.show()


In [ ]:
# 4e: PC1 vs PC2 scores plot
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(pc_scores[:,0], pc_scores[:,1],
                c=df_clean["HOMO_LUMO_Gap_eV"], cmap="viridis",
                alpha=0.6, s=20, edgecolors="none")
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)"); ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)")
ax.set_title("PCA Scores: PC1 vs PC2", fontweight="bold")
ax.axhline(y=0,color="gray",ls="--",alpha=0.3)
ax.axvline(x=0,color="gray",ls="--",alpha=0.3)
cbar = plt.colorbar(sc, ax=ax); cbar.set_label("HOMO-LUMO Gap (eV)")
# Annotate extreme PC1 points
for idx in np.argsort(np.abs(pc_scores[:,0]))[-5:]:
    ax.annotate(str(df_clean.iloc[idx]["id"]),
                (pc_scores[idx,0], pc_scores[idx,1]),
                fontsize=7, alpha=0.8, xytext=(5,5), textcoords="offset points")
plt.tight_layout(); plt.show()


In [ ]:
# 4f: Loading table and biplot
loading_df = pd.DataFrame(loadings, index=prop_names,
                          columns=[f"PC{i+1}" for i in range(loadings.shape[1])])
print("PCA Loadings (first 5 PCs):")
display(loading_df.iloc[:,:5].round(4))

print("\nStrongest PC1 loadings:")
for var in loading_df["PC1"].abs().sort_values(ascending=False).index[:5]:
    print(f"  {var:45s} {loading_df.loc[var,'PC1']:+.4f}")

print("\nStrongest PC2 loadings:")
for var in loading_df["PC2"].abs().sort_values(ascending=False).index[:5]:
    print(f"  {var:45s} {loading_df.loc[var,'PC2']:+.4f}")

# Loading biplot
sp = ["Energy","es","Polariz.","HOMO","LUMO","Inner_corr",
      "Th_corr_Eth","Th_corr_G","Entropy","Dipole","Gibbs","Enthalpy","Gap"]
fig, ax = plt.subplots(figsize=(10,8))
for i in range(len(prop_names)):
    ax.arrow(0,0, loadings[i,0], loadings[i,1],
             head_width=0.03, head_length=0.03, fc="steelblue", ec="steelblue", alpha=0.8)
    ax.text(loadings[i,0]*1.15, loadings[i,1]*1.15, sp[i], fontsize=9, ha="center", va="center")
circle = plt.Circle((0,0),1,fill=False,color="gray",ls="--",alpha=0.5)
ax.add_patch(circle)
ax.set_xlim(-1.1,1.1); ax.set_ylim(-1.1,1.1)
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)"); ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)")
ax.set_title("PCA Loading Plot: PC1 vs PC2", fontweight="bold")
ax.axhline(y=0,color="gray",alpha=0.3); ax.axvline(x=0,color="gray",alpha=0.3)
ax.set_aspect("equal"); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# 4g: Component retention and interpretation
n_kaiser = (evr > 1/len(prop_names)).sum()
n_80 = np.argmax(cvr >= 0.80) + 1
n_90 = np.argmax(cvr >= 0.90) + 1

print(f"Retention criteria:")
print(f"  Kaiser (>1 eigenvalue): {n_kaiser:2d} PCs ({cvr[n_kaiser-1]*100:.1f}%)")
print(f"  80% cumulative:         {n_80:2d} PCs ({cvr[n_80-1]*100:.1f}%)")
print(f"  90% cumulative:         {n_90:2d} PCs ({cvr[n_90-1]*100:.1f}%)")
print(f"\nDecision: Retain {n_80} components (80% threshold).")
print(f"Purpose is exploratory — understanding latent structure.")
print(f"{n_80} PCs capture {cvr[n_80-1]*100:.1f}% variance with substantial dimension reduction.")

print("\nPC1 interpretation: Molecular size axis")
print("  Dominated by total-energy quantities (Energy, Enthalpy, Gibbs) with")
print("  opposite-sign contributions from size-dependent quantities (Polarizability, Entropy).")

print("\nPC2 interpretation: Thermochemical correction axis")
print("  Dominated by Thermal_correction_to_Gibbs, Entropy, and Polarizability.")
print("  Captures thermal/entropic effects correlated with molecular flexibility.")

# Extreme points
ineg, ipos = np.argmin(pc_scores[:,0]), np.argmax(pc_scores[:,0])
print(f"\nMost negative PC1: ID={df_clean.iloc[ineg]['id']}, Energy={df_clean.iloc[ineg]['Energy_Hatree']:.1f} H")
print(f"Most positive PC1:  ID={df_clean.iloc[ipos]['id']}, Energy={df_clean.iloc[ipos]['Energy_Hatree']:.1f} H")


### PCA Findings

**Component retention:** Based on the 80% cumulative variance criterion (exact N shown in output above). Kaiser criterion and 90% threshold provide alternative reference points.

**PC1:** Molecular size axis — contrasts total-energy quantities against size-dependent quantities. Larger polymers = more negative energy + larger polarizability/entropy.

**PC2:** Thermochemical correction axis — captures thermal/entropic effects. Polymers with flexible backbones score higher.

**Unusual score-plot points:** The most extreme PC1 points represent the smallest and largest polymers (IDs annotated on plot).

**Limitations:**
1. **PCA is linear.** Non-linear relationships (e.g., Gap = |HOMO - LUMO|) spread variance across multiple PCs.
2. **Loadings are observational, not causal.** High loading != causation.
3. **2D projection hides information.** Polymers identical in PC1-PC2 may differ on PC3 (dominated by HOMO-LUMO Gap).


---
## 5. Critical-Thinking Questions


### 5.1 Dataset Structure
**Which columns are identifiers, structures, categories, or numeric analytical variables, and why does that distinction matter?**

- **Identifiers (2):** `id`, `Name` — row labels, not measurements. `id` must be excluded from PCA.
- **Chemical structures (4):** `reactant_1`, `reactant_2`, `psmiles`, `SMILES` — SMILES requiring featurization.
- **Categorical metadata (4):** `reaction_type`, `Software`, `Theory_Level`, `Source` — all constant (1 unique value each). Zero within-dataset discriminating power.
- **Numeric analytical variables (13):** The 13 quantum-chemical and thermochemical properties.

**Why it matters:** (1) Including `id` in PCA injects a spurious monotonic dimension. (2) Constant categorical columns confirm no batch effects. (3) Several numeric columns are mathematically derived (see 5.7). (4) SMILES require featurization before modeling.


### 5.2 Descriptive Statistics
**When do the mean and median describe different "typical" values, and what does that reveal about distribution shape?**

The mean and median diverge when a distribution is skewed:

- **Dipole_Debye (right-skewed):** Mean > median because a few highly polar polymers pull the mean upward. Evidence: histogram and box plot in Section 2 show a long right tail.
- **Energy_Hatree (left-skewed):** Mean < median because a few very large polymers pull the mean downward. Evidence: skewness statistic and distribution plot.

When mean ~ median (e.g., HOMO_eV), the distribution is roughly symmetric. The direction of the gap reveals the direction of skew — the median is always the more robust measure of central tendency.


### 5.3 Data Quality
**Does "zero missing values" prove that the dataset is complete? Which zero or extreme values deserve investigation?**

**No.** Pandas reports 0 NaN, but structured quality issues exist:

1. **74 failed-calculation rows (7.4%):** Energy=HOMO=LUMO=0 — physically impossible. Partial thermochemical results were written before SCF convergence check. Evidence: Section 1 audit.
2. **Gap ceiling at 10.001 eV:** Distribution terminates abruptly, not gradually decaying. Evidence: Section 2 histogram.
3. **Near-constant variables:** Thermal_correction_to_Enthalpy has only 2 unique values. All 4 categorical columns are constant.
4. **36 rows with all-zero thermochemistry:** Physically impossible (ZPE exists at 0 K).

A real audit must check for physically implausible values, near-constant columns, and pipeline artifacts — not just null cells.


### 5.4 Distribution Shapes
**Is a variable symmetric, skewed, multimodal, bounded, or concentrated at a special value?**

Evidence from Section 2 distribution plots:

- **HOMO_eV:** Approximately symmetric, unimodal. Evidence: histogram near -8.5 eV, equal tails, skewness ~0.
- **Dipole_Debye:** Right-skewed, unimodal. Evidence: peak at 3-5 D, long right tail, box-plot whisker points.
- **HOMO_LUMO_Gap_eV:** Right-skewed with hard upper bound at 10.001 eV. Evidence: abrupt cutoff in histogram.
- **Energy_Hatree:** Left-skewed. Evidence: skewness statistic, mean < median, sparse tail to -9264 H.
- **LUMO_eV:** Approximately symmetric. Evidence: histogram centered near 0 eV.
- **Isotropic_Polarizability_au:** Roughly symmetric, broad. Evidence: wide unimodal histogram, mean ~ median.


### 5.5 Box Plots and Outliers
**What do the median, box, whiskers, and points beyond the whiskers reveal? Why is a box-plot outlier not automatically an error?**

- **Median:** 50th percentile (center line).
- **Box:** Q1 to Q3 (IQR) — middle 50% of data.
- **Whiskers:** Extend to furthest point within 1.5 x IQR.
- **Points beyond whiskers:** Flagged by the 1.5xIQR heuristic.

**Why not errors:**
1. The 1.5xIQR rule is a heuristic — it flags ~0.7% of normal-distribution points by design.
2. A dipole of 14 Debye is chemically plausible (push-pull conjugated systems).
3. Low-gap polymers (~5.5 eV) are chemically meaningful — strong electron-withdrawing groups legitimately reduce the band gap.
4. The Gap ceiling at 10.001 eV is more suspicious than whisker outliers because it shows an **abrupt cutoff**, not a gradual tail.


### 5.6 Scale
**How do units and numerical ranges affect comparisons among variables?**

The 13 properties span dramatically different scales:
- Energy/Hartree-scale: range ~8,370 Hartree
- HOMO/LUMO/Gap: range ~2.8-5.5 eV (~0.1-0.2 Hartree)
- This is an ~80,000x difference in raw variance.

**Impact:** Without standardization, Energy_Hatree would dominate PC1. Electronic properties would contribute near zero. PCA would capture magnitude structure ("which variables have the biggest numbers") rather than correlation structure ("which properties move together").

**Solution:** Standardization converts to dimensionless z-scores (mean=0, std=1). Post-standardization PC1 receives diverse contributions — a composite raw-scale PCA would never produce. Evidence: Section 4 loading table.


### 5.7 Correlation
**What relationship does Pearson detect, what relationship does Spearman detect, and why can their values differ?**

- **Pearson r:** Measures **linear** relationship between raw values. Sensitive to outliers.
- **Spearman rho:** Measures **monotonic** relationship between ranks. Robust to outliers.

When |r - rho| is large: the linear assumption breaks down — the relationship is non-linear or outlier-driven. Evidence: Section 3 heatmaps and difference table.

**What the difference does NOT prove:** non-linearity (could be heteroskedasticity), causation, or functional form. It is a signal to investigate, not a conclusion.


### 5.8 Standardization
**What changes after z-score standardization, what stays the same, and why should it happen before PCA here?**

**Changes:** Mean -> 0, std -> 1, units disappear, range compresses to ~[-3,+3]. Verified numerically in Section 4.

**Stays the same:** Distribution shape (skewness, kurtosis), rank order, Pearson correlation matrix.

**Why before PCA:** Without it, Energy_Hatree (~8,370 Hartree range) dominates PC1 completely, and electronic properties contribute nothing. Standardization ensures PCA captures **correlation structure**, not magnitude structure.


### 5.9 PCA
**How many components retain a useful amount of variance? Which original properties shape PC1 and PC2?**

**Retention:** Based on 80% cumulative variance. Exact N shown in Section 4 output. Kaiser criterion and 90% threshold provide reference.

**PC1 (molecular size axis):** Dominated by total-energy quantities (Energy, Enthalpy, Gibbs) and size-dependent quantities (Polarizability, Entropy). Larger polymers = more negative energy + larger polarizability.

**PC2 (thermochemical axis):** Dominated by Thermal_correction_to_Gibbs, Entropy, and Polarizability. Captures thermal/entropic effects.

**Note:** HOMO_LUMO_Gap loads weakly on PC1/PC2 but heavily on PC3 — orthogonal to size/thermochemical axes, not unimportant. Evidence: Section 4 loading table.


### 5.10 PCA Limitations
**Does high explained variance imply prediction, causation, or scientific importance? What information can a 2D projection hide?**

**No to all three:**
1. **Prediction:** PCA is unsupervised — no target variable. Variance explained != predictive power.
2. **Causation:** Large loading = correlation, not causation.
3. **Scientific importance:** HOMO_LUMO_Gap loads heavily on PC3, not PC1/PC2. Band gap is arguably the most important property for optoelectronics, yet PCA buries it. Variance != relevance.

**What 2D projection hides:**
- Information along PC3+ (e.g., HOMO_LUMO_Gap variation).
- Non-linear relationships (Gap = |HOMO - LUMO| spreads across PCs).
- Outlier masking: extreme PC5 points invisible in PC1-PC2 plot.
- Cluster overlap: genuine 13D clusters may overlap in 2D.


---
## 6. Review Your Work (Codex Review Reflection)


### Suggestion ACCEPTED
> "Exclude the 74 failed-calculation rows from PCA and correlation analysis. These rows have physically impossible values (zero total energy, zero HOMO-LUMO gap). Including them would distort PCA by creating artificial clusters at zero."

**Evidence:** Failed rows retain non-zero values in es, Entropy, Dipole, and thermochemical corrections — proving partial results before SCF convergence check. Using 926 valid rows is more honest. 7.4% loss still leaves adequate power.


### Suggestion MODIFIED (not rejected)
> "Drop Thermal_correction_to_Enthalpy_Hatree from PCA since it has only 2 unique values."

**What I did instead:** Kept it in PCA to maintain the full 13-property specification, but documented that its near-zero variance means negligible loading and zero discriminating power. More transparent than silently dropping.

**Evidence:** Column has 964 identical values (0.00094421) and 36 zeros. Standard deviation effectively zero. Loading on PC1/PC2 is accordingly minimal (see Section 4 loading table).


---
## Bonus: Connect SVD to PCA


In [ ]:
# Starting from standardized X_scaled (926 x 13)

U_svd, S_svd, Vt_svd = np.linalg.svd(X_scaled, full_matrices=False)
pca_check = PCA()
scores_sklearn = pca_check.fit_transform(X_scaled)

print("BONUS: Connecting SVD to PCA")
print("="*60)
print()

# 1. Principal directions
print("1. X = U Sigma V^T")
print("   V^T (right singular vectors) = principal directions = PCA.components_")
print(f"   V^T shape: {Vt_svd.shape}")

# 2. Singular values -> explained variance
var_svd = (S_svd**2) / (X_scaled.shape[0]-1)
evr_svd = var_svd / var_svd.sum()
print("\n2. lambda_i = sigma_i^2/(n-1), evr = lambda_i/sum(lambda)")
print("   SVD vs sklearn PCA (first 5):")
for i in range(5):
    match = np.isclose(evr_svd[i], pca_check.explained_variance_ratio_[i])
    print(f"   PC{i+1}: SVD={evr_svd[i]:.6f}, sklearn={pca_check.explained_variance_ratio_[i]:.6f}, match={match}")

# 3. Scores
svd_scores = U_svd * S_svd
print(f"\n3. Scores: max|U*Sigma - sklearn.scores| = {np.abs(svd_scores-scores_sklearn).max():.2e}")

# 4. Loadings
print("\n4. Loadings (V^T vs sklearn.components_):")
for i in range(3):
    dot = np.dot(Vt_svd[i], pca_check.components_[i])
    sign = "same" if dot > 0 else "flipped"
    print(f"   PC{i+1}: {sign} sign (PCA directions have arbitrary sign)")

print("\nKey: PCA = SVD on centered/scaled data. V^T = directions, sigma^2/(n-1) = variance, U*Sigma = scores.")
